# Subject-Independent EEG Classification with FBGAN + CRNN-DF
## PhysioNet EEG Motor Movement/Imagery Dataset adaptation

This notebook adapts the 2023 Zhang et al. hybrid architecture from BCI Competition IV-2a `(22, 1000)` to PhysioNet EEGBCI `(64, 640)`.

**Four classes:** left fist, right fist, both fists, both feet.  
**Motor-imagery runs:** 4, 8, 12, 6, 10, 14.  
**Demonstration default:** `num_subjects_to_test = 5`.

> The original paper used subject-specific FBGAN augmentation and CRNN-DF with discriminative features. This implementation keeps the paper's methodology while making only geometry-dependent changes required by 64 channels and 640 samples.

In [1]:
# CELL 1 - Environment Setup & Library Imports
# ============================================================
# Recommended:
# pip install torch mne numpy scipy scikit-learn pandas matplotlib seaborn tqdm

import os
import random
import warnings
from dataclasses import dataclass
from pathlib import Path

import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import butter, sosfiltfilt
from sklearn.linear_model import LassoCV
from sklearn.manifold import TSNE
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings("ignore")
mne.set_log_level("ERROR")

SEED = 42

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything()

DEVICE = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)

@dataclass
class Config:
    sfreq: int = 160
    epoch_seconds: float = 4.0
    n_channels: int = 64
    n_times: int = 640
    n_classes: int = 4
    all_subjects: tuple = tuple(range(1, 110))
    num_subjects_to_test: int = 5
    data_path: str = str(Path.home() / "mne_data")
    batch_size: int = 32
    gan_batch_size: int = 5
    classifier_lr: float = 1e-4
    gan_lr: float = 1e-4
    classifier_epochs: int = 60
    gan_epochs: int = 20
    latent_dim: int = 1600
    lambda_center: float = 0.1
    center_update_every: int = 15
    center_alpha: float = 0.02
    num_workers: int = 0
    use_fbgans: bool = False
    fake_per_class: int = 10
    debug_shapes: bool = False

CFG = Config()

BANDS = [
    (1, 4), (4, 8), (8, 12), (12, 16), (16, 20),
    (20, 24), (24, 28), (28, 32), (32, 35), (35, 38),
]

CLASS_NAMES = {
    0: "Left Fist",
    1: "Right Fist",
    2: "Both Fists",
    3: "Both Feet",
}

print("Device:", DEVICE)
print("Expected input:", (1, CFG.n_channels, CFG.n_times))

Device: mps
Expected input: (1, 64, 640)


# CELL 2 - ASCII Infographics and Dimension Tracking

```text
CRNN-DF: PhysioNet adaptation

Input EEG
(B, 1, 64, 640)
        |
Spatial-temporal Conv2D
kernel=(64, 45), out=40, padding=(0, 22)
        v
(B, 40, 1, 640)
        |
BatchNorm + ELU
        |
MaxPool2D kernel=(1, 48), stride=(1, 8)
        v
(B, 40, 1, 75)
        |
squeeze spatial dimension -> (B, 75, 40)
        |
2-layer LSTM, hidden_size=64
        v
(B, 75, 64)
        |
last time step
        v
Feature vector (B, 64)
        |
Classifier -> logits (B, 4)


FBGAN: PhysioNet adaptation

z ~ N(0, I), z_dim=1600
        |
FC -> reshape (B, 256, 4, 40)
        |
ConvTranspose x 5
(B,256,4,40) -> (B,256,4,40)
               -> (B,128,8,80)
               -> (B, 64,16,160)
               -> (B, 32,32,320)
               -> (B,  1,64,640)
        |
      Fake EEG

D_phi: raw EEG discriminator
(B,1,64,640)
 -> Conv kernel=(64,15)
 -> temporal pooling / convolution
 -> FC -> real/fake

D_psi: filter-bank spatial discriminator
(B,10,64,640)
 -> Conv kernel=(64,15), one spatial collapse per band
 -> temporal pooling / convolution
 -> FC -> real/fake
```

The CRNN pooling change is `(1, 75), stride 10` -> `(1, 48), stride 8`, producing a stable sequence of 75 steps from 640 samples.

In [2]:
# ============================================================
# CELL 3 - LOCAL PHYSIONET EEGMMIDB DATA LOADER
# ============================================================
# This notebook uses the already-downloaded PhysioNet EEGMMIDB
# dataset directly from the local Mac filesystem.
#
# Expected structure:
#
# /Users/ashokvarmabevara/MtechProj/eegmmidb/
# ├── S001/
# │   ├── S001R04.edf
# │   ├── S001R06.edf
# │   ├── S001R08.edf
# │   ├── S001R10.edf
# │   ├── S001R12.edf
# │   └── S001R14.edf
# ├── S002/
# └── ...
# ============================================================

from pathlib import Path

import mne
import numpy as np
from scipy.signal import resample


# ============================================================
# DATASET CONFIGURATION
# ============================================================

DATASET_ROOT = Path(
    "/Users/ashokvarmabevara/MtechProj/eegmmidb"
)

TARGET_FS = 160
TRIAL_DURATION = 4.0

N_CHANNELS = 64
N_SAMPLES = int(TARGET_FS * TRIAL_DURATION)

LEFT_RIGHT_RUNS = [4, 8, 12]
HANDS_FEET_RUNS = [6, 10, 14]

ALL_RUNS = (
    LEFT_RIGHT_RUNS
    + HANDS_FEET_RUNS
)

CLASS_NAMES = {
    0: "Left Fist",
    1: "Right Fist",
    2: "Both Fists",
    3: "Both Feet",
}


# ============================================================
# DATASET VALIDATION
# ============================================================

def check_dataset_structure():
    """Validate the local EEGMMIDB dataset structure."""

    print("=" * 70)
    print("LOCAL PHYSIONET EEGMMIDB DATASET CHECK")
    print("=" * 70)

    print(f"Dataset path: {DATASET_ROOT}")
    print(f"Dataset exists: {DATASET_ROOT.exists()}")

    if not DATASET_ROOT.exists():
        raise FileNotFoundError(
            "Dataset folder was not found:\n"
            f"{DATASET_ROOT}"
        )

    subject_dirs = sorted(
        [
            path
            for path in DATASET_ROOT.iterdir()
            if path.is_dir()
            and path.name.startswith("S")
        ]
    )

    print(
        f"\nNumber of subject folders found: "
        f"{len(subject_dirs)}"
    )

    if len(subject_dirs) == 0:
        raise RuntimeError(
            "No subject folders were found in the dataset."
        )

    print("\nFirst 10 subject folders:")

    for subject_dir in subject_dirs[:10]:
        print(subject_dir.name)

    print("=" * 70)

    return subject_dirs


# ============================================================
# EDF FILE RESOLUTION
# ============================================================

def get_edf_file(subject_id, run_id):
    """Return the EDF path for one subject and one run."""

    subject_folder = (
        DATASET_ROOT
        / f"S{subject_id:03d}"
    )

    expected_file = (
        subject_folder
        / f"S{subject_id:03d}R{run_id:02d}.edf"
    )

    if expected_file.exists():
        return expected_file

    candidates = list(
        subject_folder.glob(
            f"*R{run_id:02d}.edf"
        )
    )

    if candidates:
        return candidates[0]

    return None


# ============================================================
# LOAD ONE EDF RUN
# ============================================================

def load_single_run(subject_id, run_id):
    """
    Load one local PhysioNet EEGMMIDB EDF recording.

    The data is resampled to 160 Hz if necessary.
    """

    edf_path = get_edf_file(
        subject_id=subject_id,
        run_id=run_id,
    )

    if edf_path is None:
        print(
            f"[WARNING] Missing EDF file: "
            f"S{subject_id:03d} R{run_id:02d}"
        )
        return None

    try:
        raw = mne.io.read_raw_edf(
            edf_path,
            preload=True,
            verbose=False,
        )

        original_fs = float(
            raw.info["sfreq"]
        )

        if not np.isclose(
            original_fs,
            TARGET_FS,
        ):
            print(
                f"Resampling S{subject_id:03d} "
                f"R{run_id:02d}: "
                f"{original_fs:.2f} Hz -> "
                f"{TARGET_FS} Hz"
            )

            raw.resample(
                TARGET_FS,
                npad="auto",
                verbose=False,
            )

        return raw

    except Exception as error:
        print(
            f"[ERROR] Failed to load "
            f"S{subject_id:03d} "
            f"R{run_id:02d}"
        )
        print(error)

        return None


# ============================================================
# RUN-SPECIFIC LABEL MAPPING
# ============================================================

def get_class_mapping(run_id):
    """
    Map PhysioNet T1/T2 annotations to the unified 4 classes.

    Runs 4, 8, 12:
        T1 -> Left Fist
        T2 -> Right Fist

    Runs 6, 10, 14:
        T1 -> Both Fists
        T2 -> Both Feet
    """

    if run_id in LEFT_RIGHT_RUNS:
        return {
            "T1": 0,
            "T2": 1,
        }

    if run_id in HANDS_FEET_RUNS:
        return {
            "T1": 2,
            "T2": 3,
        }

    raise ValueError(
        f"Unsupported run: {run_id}"
    )


# ============================================================
# EXTRACT FIXED 4-SECOND TRIALS
# ============================================================

def extract_trials_from_run(
    raw,
    subject_id,
    run_id,
):
    """
    Extract motor-imagery trials.

    Output:
        X -> (N_trials, 64, 640)
        y -> (N_trials,)
    """

    class_mapping = get_class_mapping(
        run_id
    )

    try:
        events, event_id = (
            mne.events_from_annotations(
                raw,
                verbose=False,
            )
        )

    except Exception as error:
        print(
            f"[WARNING] Annotation extraction failed "
            f"for S{subject_id:03d} "
            f"R{run_id:02d}"
        )
        print(error)

        return None, None

    annotation_by_code = {
        code: name
        for name, code in event_id.items()
    }

    selected_events = []

    for event in events:
        annotation = annotation_by_code.get(
            event[2]
        )

        if annotation in class_mapping:
            selected_events.append(event)

    if not selected_events:
        print(
            f"[WARNING] No valid motor-imagery "
            f"events found for "
            f"S{subject_id:03d} "
            f"R{run_id:02d}"
        )

        return None, None

    selected_events = np.asarray(
        selected_events,
        dtype=int,
    )

    selected_event_id = {
        name: code
        for name, code in event_id.items()
        if name in class_mapping
    }

    try:
        epochs = mne.Epochs(
            raw,
            selected_events,
            event_id=selected_event_id,
            tmin=0.0,
            tmax=(
                TRIAL_DURATION
                - 1.0 / TARGET_FS
            ),
            baseline=None,
            preload=True,
            picks="eeg",
            verbose=False,
        )

        X = epochs.get_data(
            copy=True
        )

    except Exception as error:
        print(
            f"[ERROR] Epoch extraction failed "
            f"for S{subject_id:03d} "
            f"R{run_id:02d}"
        )
        print(error)

        return None, None

    labels = []

    for event in epochs.events:
        annotation = annotation_by_code.get(
            event[2]
        )

        if annotation in class_mapping:
            labels.append(
                class_mapping[annotation]
            )

    y = np.asarray(
        labels,
        dtype=np.int64,
    )

    # Defensive shape handling.
    if X.shape[1] != N_CHANNELS:
        print(
            f"[WARNING] Expected {N_CHANNELS} EEG channels "
            f"but found {X.shape[1]} for "
            f"S{subject_id:03d} "
            f"R{run_id:02d}. Skipping run."
        )

        return None, None

    if X.shape[2] != N_SAMPLES:
        X = resample(
            X,
            N_SAMPLES,
            axis=2,
        )

    X = X.astype(
        np.float32,
        copy=False,
    )

    print(
        f"S{subject_id:03d} "
        f"R{run_id:02d} | "
        f"X={X.shape} | "
        f"Class counts="
        f"{np.bincount(y, minlength=4)}"
    )

    return X, y


# ============================================================
# LOAD ALL SIX MI RUNS FOR ONE SUBJECT
# ============================================================

def load_subject_data(subject_id):
    """
    Load runs 4, 6, 8, 10, 12, and 14 for one subject.

    Returns:
        X_subject: (N_trials, 64, 640)
        y_subject: (N_trials,)
    """

    all_trials = []
    all_labels = []

    print("\n" + "=" * 70)
    print(
        f"LOADING SUBJECT S{subject_id:03d}"
    )
    print("=" * 70)

    for run_id in ALL_RUNS:
        raw = load_single_run(
            subject_id=subject_id,
            run_id=run_id,
        )

        if raw is None:
            continue

        X_run, y_run = (
            extract_trials_from_run(
                raw=raw,
                subject_id=subject_id,
                run_id=run_id,
            )
        )

        if X_run is None:
            continue

        all_trials.append(X_run)
        all_labels.append(y_run)

    if not all_trials:
        print(
            f"[WARNING] No valid trials found "
            f"for S{subject_id:03d}"
        )

        return None, None

    X_subject = np.concatenate(
        all_trials,
        axis=0,
    )

    y_subject = np.concatenate(
        all_labels,
        axis=0,
    )

    print("\nSubject summary")
    print(
        f"Subject: S{subject_id:03d}"
    )
    print(
        f"EEG shape: {X_subject.shape}"
    )
    print(
        f"Class distribution: "
        f"{np.bincount(y_subject, minlength=4)}"
    )

    return X_subject, y_subject


# ============================================================
# LOAD MULTIPLE SUBJECTS
# ============================================================

def load_dataset(
    subject_ids,
):
    """
    Load a collection of subjects.

    Returns:
        X_all
        y_all
        subject_ids_per_trial
    """

    X_parts = []
    y_parts = []
    subject_parts = []

    for subject_id in subject_ids:
        try:
            X_subject, y_subject = (
                load_subject_data(
                    subject_id
                )
            )

            if X_subject is None:
                continue

            X_parts.append(X_subject)
            y_parts.append(y_subject)

            subject_parts.append(
                np.full(
                    len(y_subject),
                    subject_id,
                    dtype=np.int64,
                )
            )

        except Exception as error:
            print(
                f"[ERROR] Skipping "
                f"S{subject_id:03d}:"
            )
            print(error)

    if not X_parts:
        raise RuntimeError(
            "No valid subjects were loaded."
        )

    X_all = np.concatenate(
        X_parts,
        axis=0,
    )

    y_all = np.concatenate(
        y_parts,
        axis=0,
    )

    subject_ids_per_trial = (
        np.concatenate(
            subject_parts,
            axis=0,
        )
    )

    print("\n" + "=" * 70)
    print("DATASET LOADING COMPLETE")
    print("=" * 70)
    print(
        f"X shape: {X_all.shape}"
    )
    print(
        f"y shape: {y_all.shape}"
    )
    print(
        f"Loaded subjects: "
        f"{np.unique(subject_ids_per_trial)}"
    )

    return (
        X_all,
        y_all,
        subject_ids_per_trial,
    )


# ============================================================
# INITIAL DATASET TEST
# ============================================================

subject_directories = (
    check_dataset_structure()
)

X_test, y_test = load_subject_data(
    subject_id=1
)

if X_test is not None:
    assert X_test.ndim == 3
    assert X_test.shape[1] == N_CHANNELS
    assert X_test.shape[2] == N_SAMPLES

    print("\n" + "=" * 70)
    print("LOCAL DATASET TEST PASSED")
    print("=" * 70)
    print(
        f"Final test shape: {X_test.shape}"
    )
    print(
        f"Final labels shape: {y_test.shape}"
    )


LOCAL PHYSIONET EEGMMIDB DATASET CHECK
Dataset path: /Users/ashokvarmabevara/MtechProj/eegmmidb
Dataset exists: True

Number of subject folders found: 109

First 10 subject folders:
S001
S002
S003
S004
S005
S006
S007
S008
S009
S010

LOADING SUBJECT S001
S001 R04 | X=(15, 64, 640) | Class counts=[8 7 0 0]
S001 R08 | X=(15, 64, 640) | Class counts=[8 7 0 0]
S001 R12 | X=(15, 64, 640) | Class counts=[7 8 0 0]
S001 R06 | X=(15, 64, 640) | Class counts=[0 0 7 8]
S001 R10 | X=(15, 64, 640) | Class counts=[0 0 7 8]
S001 R14 | X=(15, 64, 640) | Class counts=[0 0 7 8]

Subject summary
Subject: S001
EEG shape: (90, 64, 640)
Class distribution: [23 22 21 24]

LOCAL DATASET TEST PASSED
Final test shape: (90, 64, 640)
Final labels shape: (90,)


In [3]:
# CELL 4 - Preprocessing & FBCSP + LASSO
# ============================================================

def butter_bandpass(data, low, high, sfreq, order=5):
    nyquist = sfreq / 2.0
    low = max(low / nyquist, 1e-5)
    high = min(high / nyquist, 0.999)
    sos = butter(
        order,
        [low, high],
        btype="bandpass",
        output="sos",
    )
    return sosfiltfilt(sos, data, axis=-1).astype(np.float32)


class TrainZScore:
    """Channel/time training-set normalization.

    The paper denotes normalization with sigma^2 in its equation.
    For numerical Z-score standardization, sigma is the training-set
    standard deviation, computed as sqrt(variance).
    """

    def fit(self, x):
        self.mean_ = x.mean(axis=(0, 2), keepdims=True)
        variance = x.var(axis=(0, 2), keepdims=True)
        self.std_ = np.sqrt(variance + 1e-8)
        return self

    def transform(self, x):
        return ((x - self.mean_) / self.std_).astype(np.float32)

    def fit_transform(self, x):
        return self.fit(x).transform(x)


def normalized_covariance(trial):
    covariance = trial @ trial.T
    trace = np.trace(covariance)
    return covariance / (trace + 1e-10)


class OVRFBCSP:
    """One-vs-rest CSP for each filter bank.

    Four classes x four eigenvectors = 16 candidate features per band.
    Ten bands -> 160 candidate dimensions.
    """

    def __init__(self, bands=BANDS, n_vectors_per_class=4, reg=1e-6):
        self.bands = bands
        self.n_vectors_per_class = n_vectors_per_class
        self.reg = reg
        self.filters_ = []

    def fit(self, x, y, sfreq=160):
        self.filters_ = []

        for low, high in self.bands:
            x_band = butter_bandpass(x, low, high, sfreq)
            band_filters = []

            for class_id in range(4):
                positive = x_band[y == class_id]
                negative = x_band[y != class_id]

                if len(positive) == 0 or len(negative) == 0:
                    continue

                r_pos = np.mean(
                    [normalized_covariance(t) for t in positive],
                    axis=0,
                )
                r_neg = np.mean(
                    [normalized_covariance(t) for t in negative],
                    axis=0,
                )

                composite = r_pos + r_neg
                composite += self.reg * np.eye(composite.shape[0])

                eigvals, eigvecs = np.linalg.eigh(
                    np.linalg.pinv(composite) @ r_pos
                )
                order = np.argsort(eigvals)

                low_idx = order[:self.n_vectors_per_class // 2]
                high_idx = order[-self.n_vectors_per_class // 2:]
                selected = np.concatenate([low_idx, high_idx])

                band_filters.append(eigvecs[:, selected].T)

            if band_filters:
                self.filters_.append(np.concatenate(band_filters, axis=0))
            else:
                self.filters_.append(
                    np.zeros((16, x.shape[1]), dtype=np.float32)
                )

        return self

    def transform(self, x, sfreq=160):
        features = []

        for (low, high), spatial_filters in zip(
            self.bands,
            self.filters_,
        ):
            x_band = butter_bandpass(x, low, high, sfreq)
            projected = np.einsum(
                "fc,nct->nft",
                spatial_filters,
                x_band,
            )
            variance = np.var(projected, axis=-1)
            features.append(np.log(variance + 1e-10))

        return np.concatenate(features, axis=1).astype(np.float32)


class SparseFBCSP:
    def __init__(self, bands=BANDS, random_state=SEED):
        self.csp = OVRFBCSP(bands=bands)
        self.scaler = StandardScaler()
        self.random_state = random_state
        self.selector = None
        self.selected_indices_ = None
        self.selected_filters_ = None

    def fit(self, x, y, sfreq=160):
        self.csp.fit(x, y, sfreq)
        features = self.csp.transform(x, sfreq)
        features_scaled = self.scaler.fit_transform(features)

        y_targets = np.eye(4)[y]
        coefficient_strength = np.zeros(features.shape[1])

        cv_splits = min(5, np.min(np.bincount(y)))
        cv_splits = max(cv_splits, 2)

        for class_id in range(4):
            model = LassoCV(
                cv=cv_splits,
                random_state=self.random_state,
                n_jobs=None,
                max_iter=10000,
            )
            model.fit(features_scaled, y_targets[:, class_id])
            coefficient_strength += np.abs(model.coef_)

        selected = np.flatnonzero(coefficient_strength > 1e-10)

        if len(selected) == 0:
            selected = np.argsort(coefficient_strength)[-16:]

        self.selected_indices_ = selected
        return self

    def transform(self, x, sfreq=160):
        features = self.csp.transform(x, sfreq)
        features = self.scaler.transform(features)
        return features[:, self.selected_indices_].astype(np.float32)

    def selected_spatial_filters(self):
        filters = np.concatenate(self.csp.filters_, axis=0)
        return filters, self.selected_indices_


def filter_bank_tensor(x, bands=BANDS, sfreq=160):
    """Return (N, 10, 64, 640) for D_psi."""
    bank = [
        butter_bandpass(x, low, high, sfreq)
        for low, high in bands
    ]
    return np.stack(bank, axis=1).astype(np.float32)

In [4]:
# CELL 5 - FBGAN Architecture Adapted to 64 x 640
# ============================================================

# ============================================================
# ASCII INFOGRAPHIC
#
# z (B, 1600)
#    |
# FC -> (B, 256, 4, 40)
#    |
# Deconv1 -> (B, 256, 4, 40)
# Deconv2 -> (B, 128, 8, 80)
# Deconv3 -> (B,  64,16,160)
# Deconv4 -> (B,  32,32,320)
# Deconv5 -> (B,   1,64,640)
#
# D_phi:
# (B,1,64,640)
#   -> Conv(64,15): spatial collapse
#   -> temporal conv/pooling blocks
#   -> Adaptive pooling -> FC -> sigmoid
#
# D_psi:
# (B,10,64,640), ten filter-bank signals
#   -> Conv(64,15): spatial collapse
#   -> temporal conv/pooling blocks
#   -> Adaptive pooling -> FC -> sigmoid
# ============================================================

class FBGANGenerator(nn.Module):
    def __init__(self, latent_dim=1600):
        super().__init__()
        self.latent_dim = latent_dim
        self.fc = nn.Linear(latent_dim, 256 * 4 * 40)

        self.deconv1 = nn.Sequential(
            nn.ConvTranspose2d(
                256, 256, kernel_size=(3, 3), stride=(1, 1),
                padding=(1, 1)
            ),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
        )
        self.deconv2 = nn.Sequential(
            nn.ConvTranspose2d(
                256, 128, kernel_size=(4, 4), stride=(2, 2),
                padding=(1, 1)
            ),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
        )
        self.deconv3 = nn.Sequential(
            nn.ConvTranspose2d(
                128, 64, kernel_size=(4, 4), stride=(2, 2),
                padding=(1, 1)
            ),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )
        self.deconv4 = nn.Sequential(
            nn.ConvTranspose2d(
                64, 32, kernel_size=(4, 4), stride=(2, 2),
                padding=(1, 1)
            ),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
        )
        self.deconv5 = nn.ConvTranspose2d(
            32, 1, kernel_size=(4, 4), stride=(2, 2),
            padding=(1, 1)
        )

    def forward(self, z, debug=False):
        x = self.fc(z).view(-1, 256, 4, 40)
        if debug:
            print("G FC:", x.shape)
        x = self.deconv1(x)
        if debug:
            print("G deconv1:", x.shape)
        x = self.deconv2(x)
        if debug:
            print("G deconv2:", x.shape)
        x = self.deconv3(x)
        if debug:
            print("G deconv3:", x.shape)
        x = self.deconv4(x)
        if debug:
            print("G deconv4:", x.shape)
        x = torch.tanh(self.deconv5(x))
        if debug:
            print("G output:", x.shape)
        return x


class RawEEGDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=(64, 15), padding=(0, 7)),
            nn.LeakyReLU(0.2, inplace=True),
            nn.MaxPool2d(kernel_size=(1, 4), stride=(1, 4)),
            nn.Conv2d(32, 64, kernel_size=(1, 9), padding=(0, 4)),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2, inplace=True),
            nn.MaxPool2d(kernel_size=(1, 4), stride=(1, 4)),
            nn.Conv2d(64, 128, kernel_size=(1, 7), padding=(0, 3)),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.AdaptiveAvgPool2d((1, 10)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 10, 1),
        )

    def forward(self, x, debug=False):
        if debug:
            print("D_phi input:", x.shape)
        x = self.features(x)
        if debug:
            print("D_phi features:", x.shape)
        x = self.classifier(x)
        if debug:
            print("D_phi output:", x.shape)
        return x


class FilterBankDiscriminator(nn.Module):
    def __init__(self, n_bands=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(
                n_bands, 32, kernel_size=(64, 15),
                padding=(0, 7)
            ),
            nn.LeakyReLU(0.2, inplace=True),
            nn.MaxPool2d(kernel_size=(1, 4), stride=(1, 4)),
            nn.Conv2d(32, 64, kernel_size=(1, 9), padding=(0, 4)),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2, inplace=True),
            nn.MaxPool2d(kernel_size=(1, 4), stride=(1, 4)),
            nn.Conv2d(64, 128, kernel_size=(1, 7), padding=(0, 3)),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.AdaptiveAvgPool2d((1, 10)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 10, 1),
        )

    def forward(self, x, debug=False):
        if debug:
            print("D_psi input:", x.shape)
        x = self.features(x)
        if debug:
            print("D_psi features:", x.shape)
        x = self.classifier(x)
        if debug:
            print("D_psi output:", x.shape)
        return x


@torch.no_grad()
def verify_gan_shapes(cfg=CFG):
    generator = FBGANGenerator(cfg.latent_dim).to(DEVICE)
    d_phi = RawEEGDiscriminator().to(DEVICE)
    d_psi = FilterBankDiscriminator(len(BANDS)).to(DEVICE)

    z = torch.randn(2, cfg.latent_dim, device=DEVICE)
    fake = generator(z, debug=True)
    raw_score = d_phi(fake, debug=True)
    bank = fake.squeeze(1).cpu().numpy()
    bank = filter_bank_tensor(bank, sfreq=cfg.sfreq)
    bank = torch.tensor(bank, device=DEVICE)
    fb_score = d_psi(bank, debug=True)

    assert fake.shape == (2, 1, 64, 640)
    assert raw_score.shape == (2, 1)
    assert fb_score.shape == (2, 1)

verify_gan_shapes(CFG)

G FC: torch.Size([2, 256, 4, 40])
G deconv1: torch.Size([2, 256, 4, 40])
G deconv2: torch.Size([2, 128, 8, 80])
G deconv3: torch.Size([2, 64, 16, 160])
G deconv4: torch.Size([2, 32, 32, 320])
G output: torch.Size([2, 1, 64, 640])
D_phi input: torch.Size([2, 1, 64, 640])
D_phi features: torch.Size([2, 128, 1, 10])
D_phi output: torch.Size([2, 1])
D_psi input: torch.Size([2, 10, 64, 640])
D_psi features: torch.Size([2, 128, 1, 10])
D_psi output: torch.Size([2, 1])


In [5]:
# CELL 6 - CRNN-DF Architecture Adapted to 64 x 640
# ============================================================

# ============================================================
# ASCII INFOGRAPHIC
#
# Input EEG: (B, 1, 64, 640)
#       |
# Conv2D: 1 -> 40, kernel=(64,45), padding=(0,22)
#       v
# (B, 40, 1, 640)
#       |
# BN + ELU
#       |
# MaxPool2D: kernel=(1,48), stride=(1,8)
#       v
# (B, 40, 1, 75)
#       |
# squeeze + transpose
#       v
# LSTM input: (B, 75, 40)
#       |
# 2-layer LSTM, hidden=64
#       v
# Last hidden feature: (B, 64)
#       |
# FC -> 4 classes
# ============================================================

class CRNNDF(nn.Module):
    def __init__(
        self,
        n_classes=4,
        lstm_hidden=64,
        lstm_layers=2,
        dropout=0.3,
    ):
        super().__init__()

        self.spatial_temporal = nn.Sequential(
            nn.Conv2d(
                1,
                40,
                kernel_size=(64, 45),
                padding=(0, 22),
                bias=False,
            ),
            nn.BatchNorm2d(40),
            nn.ELU(inplace=True),
            nn.MaxPool2d(
                kernel_size=(1, 48),
                stride=(1, 8),
            ),
        )

        self.lstm = nn.LSTM(
            input_size=40,
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            dropout=dropout if lstm_layers > 1 else 0.0,
        )

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(lstm_hidden, n_classes)

    def forward(self, x, debug=False):
        if x.ndim == 3:
            x = x.unsqueeze(1)

        if debug:
            print("CRNN input:", x.shape)

        x = self.spatial_temporal(x)

        if debug:
            print("After CNN/pool:", x.shape)

        x = x.squeeze(2).transpose(1, 2)

        if debug:
            print("LSTM sequence:", x.shape)

        sequence, _ = self.lstm(x)
        features = sequence[:, -1, :]
        logits = self.classifier(self.dropout(features))

        if debug:
            print("Feature vector:", features.shape)
            print("Logits:", logits.shape)

        return logits, features


@torch.no_grad()
def verify_crnn_shapes(cfg=CFG):
    model = CRNNDF(n_classes=cfg.n_classes).to(DEVICE)
    x = torch.randn(2, 1, 64, 640, device=DEVICE)
    logits, features = model(x, debug=True)
    assert logits.shape == (2, 4)
    assert features.shape == (2, 64)

verify_crnn_shapes(CFG)

CRNN input: torch.Size([2, 1, 64, 640])
After CNN/pool: torch.Size([2, 40, 1, 75])
LSTM sequence: torch.Size([2, 75, 40])
Feature vector: torch.Size([2, 64])
Logits: torch.Size([2, 4])


In [6]:
# CELL 7 - Discriminative Feature Loss & Centroid Shift
# ============================================================

class CenterDistanceLoss(nn.Module):
    """L_cen = mean(||v_i - center_yi||_2)."""

    def __init__(self, n_classes=4, feature_dim=64):
        super().__init__()
        self.register_buffer(
            "centers",
            torch.zeros(n_classes, feature_dim),
        )

    def forward(self, features, labels):
        centers_batch = self.centers[labels]
        distances = torch.norm(features - centers_batch, p=2, dim=1)
        return distances.mean()

    @torch.no_grad()
    def shift_centers(self, features, labels, alpha=0.02):
        for class_id in range(self.centers.shape[0]):
            mask = labels == class_id
            if mask.any():
                batch_center = features[mask].mean(dim=0)
                self.centers[class_id] = (
                    (1.0 - alpha) * self.centers[class_id]
                    + alpha * batch_center
                )


def joint_loss(
    logits,
    features,
    labels,
    center_loss,
    lambda_center=0.1,
):
    classification = F.cross_entropy(logits, labels)
    center = center_loss(features, labels)
    total = classification + lambda_center * center
    return total, classification.detach(), center.detach()


class EEGDataset(Dataset):
    def __init__(self, x, y):
        self.x = torch.tensor(x, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, index):
        return self.x[index], self.y[index]

In [7]:
# CELL 8 - LOSO Training Loop
# ============================================================

def train_fbcsp_on_training_only(x_train, y_train, cfg=CFG):
    extractor = SparseFBCSP()
    extractor.fit(x_train, y_train, cfg.sfreq)
    return extractor


def train_classifier(
    model,
    train_loader,
    val_loader,
    cfg=CFG,
):
    model = model.to(DEVICE)
    center_loss = CenterDistanceLoss(
        n_classes=cfg.n_classes,
        feature_dim=64,
    ).to(DEVICE)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=cfg.classifier_lr,
    )

    best_state = None
    best_accuracy = -np.inf
    history = []

    for epoch in range(1, cfg.classifier_epochs + 1):
        model.train()
        running_loss = 0.0

        all_features = []
        all_labels = []

        for x_batch, y_batch in train_loader:
            x_batch = x_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)

            optimizer.zero_grad()
            logits, features = model(
                x_batch,
                debug=False,
            )

            loss, _, _ = joint_loss(
                logits,
                features,
                y_batch,
                center_loss,
                cfg.lambda_center,
            )

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * len(y_batch)
            all_features.append(features.detach())
            all_labels.append(y_batch.detach())

        if epoch % cfg.center_update_every == 0:
            features_epoch = torch.cat(all_features)
            labels_epoch = torch.cat(all_labels)
            center_loss.shift_centers(
                features_epoch,
                labels_epoch,
                alpha=cfg.center_alpha,
            )

        model.eval()
        predictions, targets = [], []

        with torch.no_grad():
            for x_batch, y_batch in val_loader:
                x_batch = x_batch.to(DEVICE)
                logits, _ = model(x_batch)
                pred = logits.argmax(dim=1).cpu().numpy()

                predictions.extend(pred)
                targets.extend(y_batch.numpy())

        accuracy = accuracy_score(targets, predictions)
        epoch_loss = running_loss / len(train_loader.dataset)

        history.append(
            {
                "epoch": epoch,
                "train_loss": epoch_loss,
                "val_accuracy": accuracy,
            }
        )

        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }

        print(
            f"Epoch {epoch:03d}/{cfg.classifier_epochs} | "
            f"loss={epoch_loss:.4f} | val_acc={accuracy:.4f}"
        )

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history)


def run_loso(x_all, y_all, subject_ids, cfg=CFG):
    unique_subjects = np.unique(subject_ids)
    unique_subjects = unique_subjects[:cfg.num_subjects_to_test]

    fold_results = []
    predictions_records = []
    histories = []
    saved_models = {}

    for test_subject in unique_subjects:
        print("\n" + "=" * 70)
        print(f"LOSO TEST SUBJECT: {test_subject}")
        print("=" * 70)

        train_mask = subject_ids != test_subject
        test_mask = subject_ids == test_subject

        x_train_raw = x_all[train_mask]
        y_train = y_all[train_mask]
        x_test_raw = x_all[test_mask]
        y_test = y_all[test_mask]

        # Strictly fit normalization on training subjects only.
        normalizer = TrainZScore()
        x_train = normalizer.fit_transform(x_train_raw)
        x_test = normalizer.transform(x_test_raw)

        # FBCSP + LASSO is also fitted strictly on training subjects.
        print("Fitting OVR-FBCSP + LASSO...")
        sparse_fbcsp = train_fbcsp_on_training_only(
            x_train,
            y_train,
            cfg,
        )

        train_sparse_features = sparse_fbcsp.transform(
            x_train,
            cfg.sfreq,
        )
        test_sparse_features = sparse_fbcsp.transform(
            x_test,
            cfg.sfreq,
        )

        print(
            "Sparse feature shapes:",
            train_sparse_features.shape,
            test_sparse_features.shape,
        )

        train_dataset = EEGDataset(x_train, y_train)
        test_dataset = EEGDataset(x_test, y_test)

        train_loader = DataLoader(
            train_dataset,
            batch_size=cfg.batch_size,
            shuffle=True,
            num_workers=cfg.num_workers,
        )
        test_loader = DataLoader(
            test_dataset,
            batch_size=cfg.batch_size,
            shuffle=False,
            num_workers=cfg.num_workers,
        )

        model = CRNNDF(n_classes=cfg.n_classes)

        model, history = train_classifier(
            model,
            train_loader,
            test_loader,
            cfg,
        )

        histories.append(
            history.assign(test_subject=test_subject)
        )

        model.eval()
        fold_predictions = []
        fold_features = []

        with torch.no_grad():
            for x_batch, _ in test_loader:
                x_batch = x_batch.to(DEVICE)
                logits, features = model(x_batch)
                fold_predictions.append(
                    logits.argmax(dim=1).cpu().numpy()
                )
                fold_features.append(features.cpu().numpy())

        predictions = np.concatenate(fold_predictions)
        features = np.concatenate(fold_features)
        accuracy = accuracy_score(y_test, predictions)

        fold_results.append(
            {
                "test_subject": int(test_subject),
                "accuracy": accuracy,
                "n_test": len(y_test),
                "n_sparse_features": train_sparse_features.shape[1],
            }
        )

        for true_label, predicted_label, feature in zip(
            y_test,
            predictions,
            features,
        ):
            predictions_records.append(
                {
                    "test_subject": int(test_subject),
                    "true": int(true_label),
                    "pred": int(predicted_label),
                    "feature": feature,
                }
            )

        saved_models[int(test_subject)] = {
            "model": model.cpu(),
            "normalizer": normalizer,
            "fbcsp": sparse_fbcsp,
        }

        print(f"Subject {test_subject} accuracy: {accuracy:.4f}")

    fold_df = pd.DataFrame(fold_results)
    predictions_df = pd.DataFrame(predictions_records)
    history_df = pd.concat(histories, ignore_index=True)

    summary = {
        "mean_accuracy": fold_df["accuracy"].mean(),
        "std_accuracy": fold_df["accuracy"].std(ddof=1)
        if len(fold_df) > 1 else 0.0,
        "n_folds": len(fold_df),
    }

    return (
        fold_df,
        predictions_df,
        history_df,
        saved_models,
        summary,
    )


# IMPORTANT:
# Set num_subjects_to_test=5 for a demonstration.
# Increase gradually after verifying memory and runtime.
(
    FOLD_DF,
    PREDICTIONS_DF,
    HISTORY_DF,
    SAVED_MODELS,
    SUMMARY,
) = run_loso(
    X_ALL,
    Y_ALL,
    SUBJECT_IDS,
    CFG,
)

print("\nLOSO SUMMARY")
print(FOLD_DF)
print(
    f"Mean accuracy: "
    f"{SUMMARY['mean_accuracy']:.4f} ± "
    f"{SUMMARY['std_accuracy']:.4f}"
)

NameError: name 'X_ALL' is not defined

In [ ]:
# CELL 9 - Evaluation Metrics and t-SNE Visualization
# ============================================================

def plot_loso_results(fold_df, history_df):
    plt.figure(figsize=(10, 4))
    plt.bar(
        fold_df["test_subject"].astype(str),
        fold_df["accuracy"],
    )
    plt.xlabel("Held-out subject")
    plt.ylabel("Accuracy")
    plt.title("LOSO accuracy by subject")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 4))
    for subject in sorted(history_df["test_subject"].unique()):
        subset = history_df[
            history_df["test_subject"] == subject
        ]
        plt.plot(
            subset["epoch"],
            subset["val_accuracy"],
            label=f"Subject {subject}",
        )

    plt.xlabel("Epoch")
    plt.ylabel("Validation accuracy")
    plt.title("LOSO training curves")
    plt.legend()
    plt.tight_layout()
    plt.show()


def evaluate_predictions(predictions_df):
    y_true = predictions_df["true"].to_numpy()
    y_pred = predictions_df["pred"].to_numpy()

    print(
        classification_report(
            y_true,
            y_pred,
            target_names=[
                CLASS_NAMES[index] for index in range(4)
            ],
            zero_division=0,
        )
    )

    matrix = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(6, 5))
    plt.imshow(matrix)
    plt.colorbar()
    plt.xticks(range(4), [CLASS_NAMES[i] for i in range(4)],
               rotation=30, ha="right")
    plt.yticks(range(4), [CLASS_NAMES[i] for i in range(4)])
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("Confusion matrix")

    for row in range(matrix.shape[0]):
        for column in range(matrix.shape[1]):
            plt.text(
                column,
                row,
                str(matrix[row, column]),
                ha="center",
                va="center",
            )

    plt.tight_layout()
    plt.show()


def plot_tsne(predictions_df, random_state=SEED):
    features = np.stack(predictions_df["feature"].to_numpy())
    labels = predictions_df["true"].to_numpy()

    perplexity = min(30, max(5, len(features) // 5))
    perplexity = min(perplexity, len(features) - 1)

    embedding = TSNE(
        n_components=2,
        perplexity=perplexity,
        init="pca",
        learning_rate="auto",
        random_state=random_state,
    ).fit_transform(features)

    plt.figure(figsize=(8, 6))

    for class_id in range(4):
        mask = labels == class_id
        plt.scatter(
            embedding[mask, 0],
            embedding[mask, 1],
            label=CLASS_NAMES[class_id],
            alpha=0.7,
        )

    plt.title("t-SNE of CRNN-DF discriminative features")
    plt.xlabel("t-SNE 1")
    plt.ylabel("t-SNE 2")
    plt.legend()
    plt.tight_layout()
    plt.show()


plot_loso_results(FOLD_DF, HISTORY_DF)
evaluate_predictions(PREDICTIONS_DF)
plot_tsne(PREDICTIONS_DF)

print("\nFINAL SUMMARY")
print(SUMMARY)

## Implementation notes

- MNE's documented EEGBCI run mapping identifies runs 4/8/12 as left-vs-right motor imagery and 6/10/14 as hands-vs-feet motor imagery.
- Normalization, CSP fitting, and LASSO selection are fitted inside each LOSO fold using training subjects only.
- The notebook defaults to five held-out folds for a manageable demonstration. Set `num_subjects_to_test = 109` only after validating disk, RAM, and runtime.
- `use_fbgans` is left disabled by default because full subject/class-specific GAN augmentation is computationally expensive. The architecture is included and shape-tested; it can be activated in a second experiment after the CRNN-DF LOSO baseline is validated.